# w9_all.ipynb — ONE-POD full campaign (fixed-split ablations + 5-fold CV)

Single queue, 44 jobs: 14 fixed-split (dose 3/4, 2048-anchor, 7-arm nsp,
rgate2/nodoc/lead ablations) followed by 30 CV jobs (6 recipes x 5 folds).
Workers = one subprocess per GPU pulling from the shared queue; heartbeat
prints every active log's tail every 3 min; the pod STOPS ITSELF when the
queue drains. Re-run this notebook after any interruption — done jobs skip.

Wall-clock: ~60-75 h on 1xA100; divide by the GPU count of this pod.

In [ ]:
# w9_all.ipynb -- constants (single-pod campaign)
import os, subprocess

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_FS = "/workspace/w9_out"        # fixed-split results
OUT_CV = "/workspace/w9_cv_out"     # CV results

# ---- phase 1: fixed-split jobs (arm, anchor_cap, nsp, doc_lead) ----
FS_JOBS = [
    ("wcle_cegate3_icetf", 512, False, 0),
    ("wcle_cegate4_icetf", 512, False, 0),
    ("wcle_cegate2_icetf", 2048, False, 0),
    ("wcle_i2ce_icetf", 2048, False, 0),
    ("wcle_i2ce_icetf", 4096, False, 0),
    ("wcle_cegate2_icetf", 512, True, 0),
    ("wcle_cegate1_icetf", 512, True, 0),
    ("wcle_i2ce_icetf", 512, True, 0),
    ("wcle_ice_icetf", 512, True, 0),
    ("wcle_ce_cetf", 512, True, 0),
    ("wcle_arc_arctf", 512, True, 0),
    ("wcle_byol_bytf", 512, True, 0),
    ("wcle_rgate2_icetf", 512, False, 0),
    ("wcle_nodoc_i2ce_icetf", 512, False, 0),
    ("wcle_vic_cetf", 512, False, 0),   # VICReg tower I=10 V=20 C=20:
    # negative-free like BYOL but with explicit V/C anti-collapse
    ("wcle_vic2_cetf", 512, False, 0),  # C-dose ablation (C 20 -> 15)
    ("wcle_epd_v25i25c1_cetf", 512, False, 0),  # CANONICAL VICReg (all 3 terms
    # on expander output, paper weights 25/25/1) -- the FAIR baseline; vic/vic2
    # died via the centroid-collapse loophole and are not citable as VICReg
    ("wcle_epd_v20i10c20_cetf", 512, False, 0),  # canonical, OUR weights
    ("wcle_epd_v20i10c15_cetf", 512, False, 0),  # canonical, OUR weights C15
    ("wcle_byol2_bytf", 512, False, 0),  # BYOL + BN projector/3-layer BN predictor
    ("wcle_byol_bytf", 512, False, 0),   # plain BYOL clean fp W16: A/B partner
    # for byol2 (BN effect) AND for the local pool-2048 plain byol (fp effect)
    ("wcle_cegate2c_icetf", 2048, False, 0),  # champion + TRAIN-TIME centering
    # (mu-EMA subtract before L2-norm); A/B partner = cegate2_icetf@g2048
    ("wcle_i2cce_icetf", 2048, False, 0),   # I2CCE: CE + I x2 + C x1; A/B = i2ce@g2048
    ("wcle_i2ccec_icetf", 2048, False, 0),  # I2CCE + train-time centering
    # ---- view-W sweep (sentence budget per review view; 16 = historical) ----
    ("wcle_i2ce_icetf", 2048, False, 0, "clean", 48),
    ("wcle_i2ce_icetf", 2048, False, 0, "clean", 64),
    ("wcle_i2cce_icetf", 2048, False, 0, "clean", 48),
    ("wcle_i2cce_icetf", 2048, False, 0, "clean", 64),
    ("wcle_i2ccec_icetf", 2048, False, 0, "clean", 48),
    ("wcle_i2ccec_icetf", 2048, False, 0, "clean", 64),
    ("wcle_ce_cetf", 512, False, 0),               # pure-CE W16 baseline (was missing)
    ("wcle_ce_cetf", 512, False, 0, "clean", 48),
    ("wcle_ce_cetf", 512, False, 0, "clean", 64),
    ("wcle_vic_cetf", 512, False, 0, "clean", 48),
    ("wcle_vic_cetf", 512, False, 0, "clean", 64),
    ("wcle_vic2_cetf", 512, False, 0, "clean", 48),
    ("wcle_vic2_cetf", 512, False, 0, "clean", 64),
    ("wcle_byol2_bytf", 512, False, 0, "clean", 48),
    ("wcle_byol2_bytf", 512, False, 0, "clean", 64),
    ("wcle_cegate2_icetf", 512, False, 16),
    ("wcle_cegate2_icetf", 512, False, 64),
    ("wcle_cegate2_icetf", 512, False, 0, "llm"),  # pretraining-leak ablation:
    # champion with wiki_llm paraphrase docs instead of raw wiki_clean
    # single-constraint towers under the SAME paraphrase docs (user decree):
    # even leak-free wiki text cannot save CE-only / margin-only / align-only
    # -- the dual-constraint (CE+I) advantage is not a leakage artifact.
    ("wcle_ce_cetf", 512, False, 0, "llm"),
    ("wcle_arc_arctf", 512, False, 0, "llm"),
    ("wcle_byol_bytf", 512, False, 0, "llm"),
]
FS_JOBS = [j if len(j) >= 5 else (*j, "clean") for j in FS_JOBS]
FS_JOBS = [j if len(j) == 6 else (*j, 16) for j in FS_JOBS]   # + view_w
# ---- phase 2: CV jobs (recipe x fold) ----
CV_RECIPES = ["wcle_cegate2_icetf", "wcle_i2ce_icetf", "wcle_ce_cetf",
              "wcle_rgate2_icetf", "wcle_nodoc_i2ce_icetf", "wcle_ice_icetf"]
N_FOLDS = 5
FS_EPOCHS, CV_EPOCHS = 1000, 600
FULL_POOL = True    # views drawn from the ENTIRE review corpus (host RAM);
                    # False = the 2048-sentence pool (local-protocol parity)
CKPT_EVERY, FS_CKPT_SEEDS, CV_CKPT_SEEDS, TOPUP_SEEDS = 50, 3, 2, 10

def _detect_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=5).stdout.strip()
        ids = [l.strip() for l in out.splitlines() if l.strip()]
        return ids if ids else ["0"]
    except Exception:
        return ["0"]

GPUS = _detect_gpus()
os.makedirs(OUT_FS, exist_ok=True)
os.makedirs(OUT_CV, exist_ok=True)
print(f"fixed-split jobs: {len(FS_JOBS)} | cv jobs: {len(CV_RECIPES)*N_FOLDS} | gpus: {GPUS}")

In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy", "h5py"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy h5py
        break

In [ ]:
# Stage the corpus into RAM.
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "wiki_llm_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
# optional prebuilt anchor packs (wscan_gal_rev_g*.npz): stage if present
for s in sorted(src.glob("wscan_gal_rev_g*.npz")):
    d = dst / s.name
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {s.name} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# ONE-TIME (only when FULL_POOL): convert embedding_h5.h5 -> flat fp16 npy +
# meta, written into DATA_SRC on the network volume. ~150 GB, ~20-60 min.
# MULTI-MACHINE SAFE: the builder is claimed atomically; other machines WAIT
# here for the READY marker (this is the one legitimate wait in the campaign).
# The npy is written to a .tmp name and atomically renamed, so a partially
# written pool can never be mistaken for a finished one.
import os, socket, time
from pathlib import Path
import numpy as np

if FULL_POOL:
    H5 = "/workspace/stable-query-latent/game_review_data/embedding_h5.h5"
    dst_v = Path(DATA_SRC) / "full_pool_fp16.npy"
    dst_m = Path(DATA_SRC) / "full_pool_meta.npz"
    ready = Path(DATA_SRC) / "full_pool_READY"
    building = Path(DATA_SRC) / "full_pool_BUILDING"
    me = socket.gethostname() + ":" + os.environ.get("RUNPOD_POD_ID", "?")

    def _claim_build():
        try:
            fd = os.open(building, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, me.encode()); os.close(fd)
            return True
        except FileExistsError:
            return building.read_text().strip() == me

    # ADOPT a pool completed by the pre-marker code: open_memmap preallocates
    # the full file, so size proves nothing -- verify content instead (real
    # embeddings are never all-zero; an interrupted build has a zero tail).
    if not ready.exists() and dst_v.exists() and dst_m.exists():
        try:
            v = np.load(dst_v, mmap_mode="r")
            probes = [v.shape[0] - 1, v.shape[0] - 2,
                      v.shape[0] * 3 // 4, v.shape[0] // 2]
            if all(float(np.abs(v[i]).sum()) > 0 for i in probes):
                ready.write_text(f"{me} adopted {v.shape[0]}")
                print("adopted pre-existing COMPLETE pool (content-verified)")
            else:
                print("pre-existing pool is INCOMPLETE (zero tail) -- rebuilding")
            del v
        except Exception as e:
            print("adoption check failed:", e)

    if ready.exists():
        print("full pool already prepared:", dst_v)
    elif _claim_build():
        import h5py
        assert Path(H5).exists(), f"{H5} not found on the volume"
        t0 = time.time()
        tmp_v = dst_v.with_suffix(".npy.tmp")
        with h5py.File(H5, "r") as h:
            N = h["vectors"].shape[0]
            np.savez(dst_m,
                     game_review_offsets=h["game_review_offsets"][:],
                     review_offsets=h["review_offsets"][:],
                     game_names=np.array([g.decode() if isinstance(g, bytes) else str(g)
                                          for g in h["game_names"][:]], object))
            out = np.lib.format.open_memmap(tmp_v, mode="w+", dtype=np.float16,
                                            shape=(N, 1024))
            B = 1_000_000
            for i in range(0, N, B):
                out[i:i+B] = h["vectors"][i:i+B]
                print(f"  {i+min(B, N-i):,}/{N:,} [{time.time()-t0:.0f}s]", flush=True)
            out.flush()
            del out
        os.replace(tmp_v, dst_v)                    # atomic finalize
        ready.write_text(f"{me} {N}")
        building.unlink(missing_ok=True)
        print(f"full pool ready in {(time.time()-t0)/60:.1f} min")
    else:
        owner = building.read_text().strip() if building.exists() else "?"
        print(f"full pool being built by {owner} -- waiting for READY ...", flush=True)
        t0 = time.time()
        while not ready.exists():
            time.sleep(30)
            if int(time.time() - t0) % 300 < 30:
                print(f"  still waiting [{(time.time()-t0)/60:.0f} min]", flush=True)
        print("READY detected; proceeding.")


In [ ]:
# Stage the 150 GB full pool onto FAST LOCAL storage (thread-parallel copy,
# pattern from Pod/h5_staging.py). Network-volume random reads are slow; one
# sequential parallel copy (~5-15 min) buys RAM/NVMe-speed sampling for the
# whole campaign. Falls back to the volume mmap if no local space is found.
import os, sys
from pathlib import Path
FULL_POOL_PATH = ""
if FULL_POOL:
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from Pod.h5_staging import parallel_copy

    src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
    src_m = Path(DATA_SRC) / "full_pool_meta.npz"
    need = src_v.stat().st_size + (5 << 30)

    def _free(p):
        st = os.statvfs(p)
        return st.f_bavail * st.f_frsize

    dest_dir = None
    for cand in ("/dev/shm", "/root/data", "/root"):
        Path(cand).mkdir(parents=True, exist_ok=True)
        if _free(cand) > need:
            dest_dir = Path(cand)
            break
    if dest_dir is None:
        print("WARNING: no local space for the full pool -- workers will mmap "
              "the NETWORK VOLUME copy (slow first pass).")
        FULL_POOL_PATH = str(src_v)
    else:
        dst_v = dest_dir / "full_pool_fp16.npy"
        if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
            print("local full pool already staged:", dst_v)
        else:
            import time
            t0 = time.time()
            tmp = dst_v.with_name(dst_v.name + ".copying")
            print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} "
                  f"(8 threads) ...", flush=True)
            parallel_copy(src_v, tmp, workers=8)
            os.replace(tmp, dst_v)
            print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
        import shutil
        shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
        FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH or "(disabled)")

In [ ]:
# ONE queue for both phases (fixed-split first, then CV), heartbeat included.
import os, queue, socket, subprocess, threading, time
from pathlib import Path

# ---- multi-machine safety: atomic job claims on the shared volume ----
# Exclusive-create of claims/<name>.claim before a job runs (lightweight form
# of training.ipynb's VM coordination). Same-host reclaim allowed (crash
# restart resumes); another machine's claim honored unless STALE (claimer
# died > CLAIM_STALE_H hours ago) -- then taken over.
HOST = socket.gethostname() + ":" + os.environ.get("RUNPOD_POD_ID", "?")
CLAIM_STALE_H = 12

def try_claim(claim_dir, nm):
    claim_dir.mkdir(parents=True, exist_ok=True)
    cl = claim_dir / f"{nm}.claim"
    try:
        fd = os.open(cl, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        os.write(fd, f"{HOST} {time.time():.0f}".encode())
        os.close(fd)
        return True
    except FileExistsError:
        try:
            owner, ts = cl.read_text().split()
            if owner == HOST:
                return True                      # our own earlier claim (restart)
            if time.time() - float(ts) > CLAIM_STALE_H * 3600:
                cl.write_text(f"{HOST} {time.time():.0f}")   # stale takeover
                print(f"[claim] took over stale {nm} (was {owner})", flush=True)
                return True
        except Exception:
            pass
        return False

def _monitor(log_dirs, stop_evt, period=180):
    seen = {}
    while not stop_evt.wait(period):
        for ld in log_dirs:
            for lg in sorted(ld.glob("*.log")):
                try:
                    sz = lg.stat().st_size
                    if seen.get(str(lg)) == sz:
                        continue
                    seen[str(lg)] = sz
                    with open(lg, "rb") as fh:
                        fh.seek(max(0, sz - 400))
                        tail = fh.read().decode(errors="ignore").strip().splitlines()
                    if tail:
                        print(f"[beat] {lg.name}: {tail[-1]}", flush=True)
                except Exception:
                    pass

jobs = queue.Queue()
n_jobs = 0
for arm, cap, nsp, lead, wsrc, vw in FS_JOBS:
    nm = (f"w9_{arm}" + (f"_g{cap}" if cap != 512 else "")
          + ("_nsp" if nsp else "") + (f"_ld{lead}" if lead else "")
          + ("_wllm" if wsrc == "llm" else "") + (f"_w{vw}" if vw != 16 else ""))
    if (Path(OUT_FS) / f"ft4var_{nm}{'_fp' if FULL_POOL else ''}_best.json").exists():
        print(f"[skip] fs {nm}")
        continue
    jobs.put(("fs", nm, arm, cap, nsp, lead, wsrc, vw)); n_jobs += 1
for r in CV_RECIPES:
    for k in range(N_FOLDS):
        if (Path(OUT_CV) / f"ft4var_w9cv_{r}_fold{k}{'_fp' if FULL_POOL else ''}_best.json").exists():
            print(f"[skip] cv {r}/fold{k}")
            continue
        jobs.put(("cv", f"w9cv_{r}_fold{k}", r, k)); n_jobs += 1

log_fs = Path(OUT_FS) / "logs"; log_fs.mkdir(exist_ok=True)
log_cv = Path(OUT_CV) / "logs"; log_cv.mkdir(exist_ok=True)
fails = []

def run_job(gpu, job):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu)
    if job[0] == "fs":
        _, nm, arm, cap, nsp, lead, wsrc, vw = job
        sfx = (("_nsp" if nsp else "") + (f"_ld{lead}" if lead else "")
               + ("_wllm" if wsrc == "llm" else "") + (f"_w{vw}" if vw != 16 else ""))
        log = log_fs / f"{arm}_g{cap}{sfx}.log"
        cmd = ["python", "-u", os.path.join(REPO, "Pod/w9_a100_worker.py"),
               "--data-dir", DATA_DIR, "--out-dir", OUT_FS, "--repo", REPO,
               "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(FS_EPOCHS), "--ckpt-every", str(CKPT_EVERY),
               "--ckpt-seeds", str(FS_CKPT_SEEDS), "--topup-seeds", str(TOPUP_SEEDS)]
        if nsp:
            cmd.append("--no-sp-view")
        if lead:
            cmd += ["--doc-lead", str(lead)]
        if wsrc == "llm":
            cmd += ["--wiki-src", "llm"]
        if vw != 16:
            cmd += ["--view-w", str(vw)]
        if FULL_POOL:
            cmd += ["--full-pool", "--full-pool-path", FULL_POOL_PATH]
        tag = f"fs {arm}@g{cap}{sfx}"
    else:
        _, nm, r, k = job
        log = log_cv / f"{r}_fold{k}.log"
        cmd = ["python", "-u", os.path.join(REPO, "Pod/w9_cv_worker.py"),
               "--data-dir", DATA_DIR, "--out-dir", OUT_CV, "--repo", REPO,
               "--arm", r, "--fold", str(k), "--n-folds", str(N_FOLDS),
               "--epochs", str(CV_EPOCHS), "--ckpt-every", str(CKPT_EVERY),
               "--ckpt-seeds", str(CV_CKPT_SEEDS), "--topup-seeds", str(TOPUP_SEEDS)]
        if FULL_POOL:
            cmd += ["--full-pool", "--full-pool-path", FULL_POOL_PATH]
        tag = f"cv {r}/fold{k}"
    print(f"[gpu{gpu}] start {tag}", flush=True)
    t0 = time.time()
    with open(log, "w") as fh:
        p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env)
    if p.returncode != 0:
        fails.append((tag, str(log)))
    print(f"[gpu{gpu}] " + ("ok" if p.returncode == 0 else "FAIL")
          + f" {tag} [{(time.time()-t0)/60:.1f} min]", flush=True)

def worker(gpu):
    while True:
        try:
            job = jobs.get_nowait()
        except queue.Empty:
            return
        cdir = (Path(OUT_FS) if job[0] == "fs" else Path(OUT_CV)) / "claims"
        if not try_claim(cdir, job[1]):
            print(f"[claim] {job[1]} held by another machine -- skipped", flush=True)
            continue
        run_job(gpu, job)

stop_evt = threading.Event()
mon = threading.Thread(target=_monitor, args=([log_fs, log_cv], stop_evt), daemon=True)
mon.start()
threads = [threading.Thread(target=worker, args=(g,)) for g in GPUS]
t0 = time.time()
for t in threads: t.start()
for t in threads: t.join()
stop_evt.set()
print(f"CAMPAIGN finished in {(time.time()-t0)/3600:.1f} h; {n_jobs} run, {len(fails)} failed")
for tag, log in fails:
    print("  FAILED:", tag, "->", log)

In [ ]:
# Aggregate both phases.
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]

print("========= fixed-split bests =========")
for j in sorted(Path(OUT_FS).glob("ft4var_*_best*.json")):
    d = json.loads(j.read_text())
    runs = d["per_seed"]
    m4 = np.mean([np.mean([r[v]["h1"] for r in runs]) for v in VORD])
    print(f"{j.stem} (ep{d.get('best_ep','?')}): "
          + " ".join(f"{v}:{np.mean([r[v]['h1'] for r in runs]):.3f}" for v in VORD)
          + f" m4:{m4:.3f}")

print("\n========= 5-fold CV =========")
for f in sorted(Path(OUT_CV).glob("w9cv_frozen_fold*.json")):
    d = json.loads(f.read_text())
    print("  " + f.stem + ": " + " ".join(f"{v}:{d[v]['h1']:.3f}" for v in VORD))
for r in CV_RECIPES:
    per_fold = []
    for k in range(N_FOLDS):
        j = Path(OUT_CV) / f"ft4var_w9cv_{r}_fold{k}_best.json"
        if not j.exists():
            continue
        runs = json.loads(j.read_text())["per_seed"]
        row = {v: np.mean([x[v]["h1"] for x in runs]) for v in VORD}
        row["m4"] = np.mean([np.mean([x[v]["h1"] for x in runs]) for v in VORD])
        per_fold.append(row)
    if not per_fold:
        continue
    print(f"\n{r} ({len(per_fold)} folds)")
    for kf in ("neutral", "noname", "m4"):
        vals = [pf[kf] for pf in per_fold]
        print(f"  {kf:8s} {np.mean(vals):.3f} +- {np.std(vals):.3f}")

In [ ]:
# AUTO-STOP: stop THIS pod when the campaign has finished. Results are on the
# network volume. Set AUTO_STOP=False to keep the pod alive.
# completeness audit BEFORE stopping: anything still missing is either being
# run by ANOTHER machine right now, or was stale-claimed by a dead pod (rerun
# w9_all later to sweep it up). This machine never idles waiting for others.
from pathlib import Path as _P
_missing = []
for arm, cap, nsp, lead, wsrc, vw in FS_JOBS:
    nm = (f"w9_{arm}" + (f"_g{cap}" if cap != 512 else "") + ("_nsp" if nsp else "")
          + (f"_ld{lead}" if lead else "") + ("_wllm" if wsrc == "llm" else "")
          + (f"_w{vw}" if vw != 16 else "") + ("_fp" if FULL_POOL else ""))
    if not (_P(OUT_FS) / f"ft4var_{nm}_best.json").exists():
        _missing.append(nm)
for r in CV_RECIPES:
    for k in range(N_FOLDS):
        nmc = f"w9cv_{r}_fold{k}" + ("_fp" if FULL_POOL else "")
        if not (_P(OUT_CV) / f"ft4var_{nmc}_best.json").exists():
            _missing.append(nmc)
if _missing:
    print(f"AUDIT: {len(_missing)} job(s) not finished on the volume "
          f"(another machine's share, or stale claims of a dead pod):")
    for m in _missing:
        print("   -", m)
else:
    print("AUDIT: campaign COMPLETE -- every job has its _best.json.")

AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs kept; stopping anyway.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")